In [105]:
# Importing all the libraries I'll need.

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import VotingClassifier

In [106]:
# Getting both the data and the test data?

data = pd.read_csv("train.csv")
test_data = pd.read_csv("test.csv")

In [107]:
# Creating a copy of the datat to prevent altering the original(best practice)

data_copy = data.copy()
test_data_copy = test_data.copy()

In [108]:
# Quickly preview the data

data_copy.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [109]:
# For this worklow- I'm dropping columns that would confuse the model

data_copy = data_copy.drop(["Cabin", "Ticket", "Name", "PassengerId"], axis=1)

In [110]:
# both the Fare and Age column contained null values and are filled with mean()

data_copy["Age"] = data_copy["Age"].fillna(data_copy["Age"].mean()).round().astype(int)
data_copy["Fare"] = data_copy["Fare"].fillna(data_copy["Fare"].mean()).round().astype(int)

In [111]:
# Regestering the Pclass as a (category) column and not an (int) column

data_copy["Pclass"]= data_copy["Pclass"].astype("category")

In [112]:
# Separating the categories within the column to remove the idea of distance by the model

data_copy_encoded1 = pd.get_dummies(data_copy, columns=["Pclass"], prefix="Pclass")

In [113]:
# One hot encoding

data_copy_encoded2 = pd.get_dummies(data_copy_encoded1, columns=["Embarked", "Sex"])

In [148]:
data_copy_encoded2.to_csv("cleaned_train.csv", index=False)

In [115]:
# Dropping columns that are inconsistent
test_data_copy = test_data_copy.drop(["Cabin", "Ticket","Name","PassengerId"], axis=1)

# Fill the Data with the mean value of the Age Column
test_data_copy["Age"] = test_data_copy["Age"].fillna(test_data_copy["Age"].mean()).round().astype(int)

# One hot encoding the Pclass column
test_data_copy["Pclass"]= test_data_copy["Pclass"].astype("category")
test_data_copy_encoded1 = pd.get_dummies(test_data_copy, columns=["Pclass"], prefix="Pclass")

# One hot encoding the other columns
test_data_copy_encoded2 = pd.get_dummies(test_data_copy_encoded1, columns=["Embarked", "Sex"])



# Creating a preview to make sure test data looks like train data
# test_data_copy_encoded2.to_csv("Titanic_Test_Preview.csv", index=False)

In [116]:
# Separating the probable cause from the effects
x_train = data_copy_encoded2.drop(["Survived"], axis=1 )

y_train = data_copy_encoded2["Survived"]

# Creating a Validation and train set
x = x_train
y = y_train

x_tr, x_val, y_tr, y_val = train_test_split(
    x,y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Assigning the model
model = LogisticRegression(max_iter=5000)
model.fit(x_tr, y_tr)



,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,5000
,multi_class,'deprecated'


In [117]:
# Testing out the model and assigning it to a variable

y_val_pred = model.predict(x_val)

In [118]:
# checking how accurate the prediction was with respect to the survived in the train data set
accuracy = accuracy_score(y_val, y_val_pred)
error_percentage = (1-accuracy) * 100

print("Accuracy: ", accuracy )
print("Error: " ,error_percentage)

Accuracy:  0.7988826815642458
Error:  20.11173184357542


In [119]:
# trying the tree model for a better decision

tree_model = DecisionTreeClassifier(max_depth= 9, random_state=42 )

In [120]:
tree_model.fit(x_tr, y_tr)

,criterion,'gini'
,splitter,'best'
,max_depth,9
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,42
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [121]:
y_val_tree_pred = tree_model.predict(x_val)

In [122]:
tree_accuracy = accuracy_score(y_val, y_val_tree_pred)
error_percentage_tree = (1- tree_accuracy) *100

print("Accuracy Score", tree_accuracy)
print("Error Percentage", error_percentage_tree)

Accuracy Score 0.7821229050279329
Error Percentage 21.78770949720671


In [133]:
forest_model = RandomForestClassifier(max_depth=20, n_estimators=41, random_state=42)

In [134]:
forest_model.fit(x_tr, y_tr)

,n_estimators,41
,criterion,'gini'
,max_depth,20
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [135]:
y_val_forest_pred = forest_model.predict(x_val)

In [136]:

forest_accuracy = accuracy_score(y_val, y_val_forest_pred)
error_percentage_forest = (1- forest_accuracy) *100

print("Accuracy Score", forest_accuracy)
print("Error Percentage", error_percentage_forest)

Accuracy Score 0.8324022346368715
Error Percentage 16.759776536312852


In [137]:
# This lets to models argue it out to reach a better conclusion

vote_model = VotingClassifier(estimators=[ 
                              ("logistic", model),
                              ("forest", forest_model)],
                              voting= "hard"
    )

In [138]:
vote_model.fit(x_tr, y_tr)

,estimators,"[('logistic', ...), ('forest', ...)]"
,voting,'hard'
,weights,None
,n_jobs,None
,flatten_transform,True
,verbose,False
,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True


In [139]:
    y_val_vote_pred = vote_model.predict(x_val)

In [140]:
vote_accuracy = accuracy_score(y_val, y_val_vote_pred)
error_percentage_vote = (1- vote_accuracy) *100

print("Accuracy Score", vote_accuracy)
print("Error Percentage", error_percentage_vote)

Accuracy Score 0.8156424581005587
Error Percentage 18.43575418994413


In [141]:
# This is to check what best combination of estimators and depth lead to more accurate results

# best_acc = 0
# best_params = {}

# for depth in range(5,50,5):
#     for n in range(1,500, 20):
#         forest_loop_model = RandomForestClassifier(max_depth=depth, n_estimators= n, random_state=42)
#         forest_loop_model.fit(x_tr, y_tr)
#         y_val_loop_forest_pred = forest_loop_model.predict(x_val)
#         acc = accuracy_score(y_val, y_val_loop_forest_pred)

#         if acc > best_acc:
#             best_acc = acc
#             best_params = {"max_depth": depth, "n_estimators": n}

# print("Best Accuracy: ", best_acc)
# print("Best params: ", best_params)

In [142]:
# I realized later the Fare had a singular null value

x_test = test_data_copy_encoded2[x_train.columns]
x_test["Fare"] = x_test["Fare"].fillna(x_test["Fare"].mean()).round().astype(int)



In [149]:
x_test.to_csv("cleaned_test.csv", index=False)

In [145]:
# Testing every model for the kaggle competition

y_val_vote_pred_test = vote_model.predict(x_test)
y_val_forest_pred_test = forest_model.predict(x_test)
y_val_tree_pred_test = tree_model.predict(x_test)
y_val_pred = model.predict(x_test)

In [146]:
# Changing the prediction to a data frame

submission = pd.DataFrame({"PassengerId" : test_data["PassengerId"],
                           "Survived" : y_val_forest_pred_test })

In [147]:
# Creating a CSV File I can submit

submission.to_csv("submission.csv", index=False)